In [ ]:
%pip install -q langchain langchain-aws langgraph boto3 python-dotenv

# Week 4 Thursday — Tools, Agents & Memory

Yesterday we learned how `init_chat_model()` gives us a **provider-agnostic** way to talk to any LLM. Today we give that model **capabilities**: tools it can call, an agent loop that decides *when* to call them, and memory so it can hold a real conversation.

This notebook combines material from two source days into a single progression.

## Learning Objectives

By the end of this notebook you will be able to:

- Create custom tools with the `@tool` decorator and write docstrings that drive correct routing
- Build agents with `create_agent()` and understand the required `name` parameter
- Test tools independently with `.invoke()` before wiring them into an agent
- Add short-term memory via `InMemorySaver` and manage thread-based conversations
- Inspect agent state to debug conversation flow

## Roadmap

| Stage | Topic | Key Ideas |
|-------|-------|-----------|
| 1 | Tool Creation with `@tool` Decorator | `@tool`, docstrings as routing instructions, type hints, Pydantic inputs |
| 2 | Building Agents with `create_agent()` | Agentic loop, `model` / `tools` / `name`, system prompts, deprecated patterns |
| 3 | Testing Tools Independently | `.invoke()`, edge cases, error handling, validation before integration |
| 4 | Short-Term Memory & Persistence | `InMemorySaver`, `thread_id`, thread isolation, state inspection |

---
## Stage 1 — Tool Creation with the `@tool` Decorator

Tools are what transform a chatbot into an **agent**. Without tools the model can only generate text; with tools it can search databases, call APIs, do math — anything a Python function can do.

The `@tool` decorator from `langchain_core.tools` converts any Python function into something an agent can call. Three things matter:

1. **Import** — `from langchain_core.tools import tool`
2. **Docstring** — the agent reads this to decide *when* to use the tool
3. **Type hints** — LangChain uses them to build the parameter schema

In [ ]:
from langchain_core.tools import tool


@tool
def greet_user(name: str) -> str:
    """Greet a user by name. Use when someone asks to be greeted or says hello."""
    return f"Hello, {name}! Welcome to the LangChain demo."


print("Tool name:       ", greet_user.name)
print("Tool description:", greet_user.description)

result = greet_user.invoke({"name": "Alice"})
print("\nDirect call result:", result)

In [ ]:
@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together. Use for any addition or sum calculation."""
    return a + b


@tool
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two numbers. Use for multiplication or product calculations."""
    return a * b


print("add_numbers(5, 3)     ->", add_numbers.invoke({"a": 5, "b": 3}))
print("multiply_numbers(5, 3) ->", multiply_numbers.invoke({"a": 5, "b": 3}))

### Why Docstrings Are Critical

The docstring is your tool's **identity** to the agent. The model never sees the function body — it only sees the name and the docstring. A vague description like `"Process data."` gives the agent nothing to work with, while a specific one like `"Convert text to uppercase. Use when asked to capitalize, shout, or make text all caps."` tells it exactly when to reach for that tool.

Good docstrings answer:
1. **What** does this tool do?
2. **When** should the agent use it?
3. *(Optional)* **When should it NOT** use it?

In [ ]:
@tool
def bad_tool(data: str) -> str:
    """Process data."""
    return data.upper()


@tool
def good_tool(text: str) -> str:
    """Convert text to uppercase. Use when asked to capitalize, shout, or make text all caps."""
    return text.upper()


print("BAD  description:", bad_tool.description)
print("GOOD description:", good_tool.description)
print("\nWhich would an agent route to correctly?")

### Different Parameter Types

Tools can accept optional parameters, lists, and even structured Pydantic inputs. Always return **strings** for the best agent compatibility — the model interprets them more reliably than raw ints or dicts.

In [ ]:
from typing import Optional, List
from pydantic import BaseModel


@tool
def search_documents(
    query: str,
    max_results: int = 5,
    category: Optional[str] = None,
) -> str:
    """
    Search the document database for relevant content.

    Use when asked to find, search, or look up documents.
    Can optionally filter by category (e.g., 'technical', 'legal', 'hr').
    """
    result = f"Searching for '{query}'"
    if category:
        result += f" in category '{category}'"
    result += f" (max {max_results} results)"
    return result


print(search_documents.invoke({"query": "LangChain"}))
print(search_documents.invoke({"query": "LangChain", "max_results": 10}))
print(search_documents.invoke({"query": "LangChain", "category": "technical"}))

In [ ]:
class OrderDetails(BaseModel):
    product_name: str
    quantity: int
    price_per_unit: float


@tool
def calculate_order_total(order: OrderDetails) -> str:
    """
    Calculate the total price for an order.

    Use when asked to calculate, compute, or find the total for a product order.
    Requires product name, quantity, and price per unit.
    """
    total = order.quantity * order.price_per_unit
    return f"Order: {order.quantity}x {order.product_name} @ ${order.price_per_unit:.2f} = ${total:.2f}"


print(calculate_order_total.invoke(
    {"order": {"product_name": "Widget", "quantity": 5, "price_per_unit": 19.99}}
))

---
## Stage 2 — Building Agents with `create_agent()`

A tool by itself just sits there — it needs an **agent** to decide when to call it. `create_agent()` from `langchain.agents` is the LangChain v1.0 standard. It wires a model to a list of tools and runs the **agentic loop** (Observe → Reason → Act → repeat) automatically.

Key parameters:

| Parameter | Required? | Purpose |
|-----------|-----------|--------|
| `model` | Yes | Provider string, e.g. `"bedrock:us.amazon.nova-2-lite-v1:0"` |
| `tools` | Yes | List of `@tool`-decorated functions |
| `name` | Yes | Unique name for tracing & debugging |
| `system_prompt` | No | Instructions that shape agent behaviour |
| `checkpointer` | No | Memory backend (covered in Stage 4) |

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city. Use when asked about weather conditions."""
    weather_data = {
        "new york": "Sunny, 72°F",
        "london": "Rainy, 55°F",
        "tokyo": "Cloudy, 68°F",
        "paris": "Partly cloudy, 65°F",
    }
    return weather_data.get(city.lower(), f"Weather data not available for {city}")


@tool
def get_time(timezone: str) -> str:
    """Get the current time in a timezone. Use when asked about time."""
    times = {
        "est": "3:00 PM EST",
        "pst": "12:00 PM PST",
        "gmt": "8:00 PM GMT",
        "jst": "5:00 AM JST",
    }
    return times.get(timezone.lower(), f"Time data not available for {timezone}")


weather_agent = create_agent(
    model="bedrock:us.amazon.nova-2-lite-v1:0",
    tools=[get_weather, get_time],
    system_prompt="You are a helpful assistant that provides weather and time information.",
    name="weather_time_agent",
)

print("Agent created: weather_time_agent")
print("Tools:", [t.name for t in [get_weather, get_time]])

### Invoking the Agent

You always invoke with `{"messages": [{"role": "user", "content": "..."}]}`. The result comes back in `result["messages"][-1].content`.

In [ ]:
result = weather_agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in New York?"}]}
)
print("User:  What's the weather in New York?")
print("Agent:", result["messages"][-1].content)

print()

result = weather_agent.invoke(
    {"messages": [{"role": "user", "content": "What time is it in Tokyo (JST)?"}]}
)
print("User:  What time is it in Tokyo (JST)?")
print("Agent:", result["messages"][-1].content)

### System Prompts Guide Behaviour

The system prompt tells the agent what role to play, how to respond, and when to use tools. Two agents with the same tools but different prompts can behave very differently.

In [ ]:
@tool
def lookup_info(topic: str) -> str:
    """Look up information about a topic. Use for any knowledge queries."""
    return f"Information about {topic}: This is sample data for demonstration."


formal_agent = create_agent(
    model="bedrock:us.amazon.nova-2-lite-v1:0",
    tools=[lookup_info],
    system_prompt="You are a formal business assistant. Use professional language and be concise.",
    name="formal_business_agent",
)

casual_agent = create_agent(
    model="bedrock:us.amazon.nova-2-lite-v1:0",
    tools=[lookup_info],
    system_prompt="You are a friendly casual assistant. Keep it light and fun.",
    name="casual_friendly_agent",
)

query = {"messages": [{"role": "user", "content": "Tell me about Python programming."}]}

print("=== Formal Agent ===")
print(formal_agent.invoke(query)["messages"][-1].content[:300])

print("\n=== Casual Agent ===")
print(casual_agent.invoke(query)["messages"][-1].content[:300])

### Agent Naming Best Practices

The `name` parameter is **required** in LangChain v1.0. Good names make debugging, LangSmith traces, and multi-agent systems far easier.

| Do | Don't |
|----|-------|
| `snake_case` | `PascalCase` or `kebab-case` |
| Descriptive: `order_tracking_agent` | Generic: `agent1` |
| Indicate purpose at a glance | Use cryptic abbreviations |

**Deprecated patterns to avoid:**
- `create_react_agent` — replaced by `create_agent()`
- LCEL chains (`prompt | llm | tools`) — removed in v1.0
- `initialize_agent` — removed
- Omitting the `name` parameter

---
## Stage 3 — Testing Tools Independently

When your agent gives a wrong answer, is the bug in the **tool** or in the **agent's reasoning**? You can't tell unless you've tested the tool on its own first.

The testing hierarchy:

```
Level 1  →  Direct function call      (tool.func(args))
Level 2  →  Tool invocation            (tool.invoke({...}))
Level 3  →  Integration with agent     (agent.invoke(...))
```

Problems at Level 1 cascade upward. Always start at the bottom.

In [ ]:
@tool
def calculate_discount(original_price: float, discount_percent: float) -> str:
    """
    Calculate the discounted price.

    Use when asked to apply a discount, calculate savings, or find sale prices.
    """
    discounted = original_price * (1 - discount_percent / 100)
    savings = original_price - discounted
    return (
        f"Original: ${original_price:.2f}, "
        f"Discount: {discount_percent}%, "
        f"Final: ${discounted:.2f}, "
        f"You save: ${savings:.2f}"
    )


print("--- Normal inputs ---")
print(calculate_discount.invoke({"original_price": 100.0, "discount_percent": 20.0}))
print(calculate_discount.invoke({"original_price": 49.99, "discount_percent": 15.0}))

print("\n--- Edge cases ---")
print("0% discount: ", calculate_discount.invoke({"original_price": 100.0, "discount_percent": 0.0}))
print("100% discount:", calculate_discount.invoke({"original_price": 100.0, "discount_percent": 100.0}))
print("Negative:    ", calculate_discount.invoke({"original_price": 100.0, "discount_percent": -20.0}))

### Error Handling in Tools

A tool should **never crash** when given bad input. If it raises an exception the agent gets a raw traceback instead of a helpful message. Return a clear error string so the agent can explain the problem or retry.

In [ ]:
@tool
def bad_divide(a: float, b: float) -> float:
    """Divide two numbers. No error handling!"""
    return a / b


@tool
def good_divide(a: float, b: float) -> str:
    """Divide two numbers safely. Returns error message if division is not possible."""
    if b == 0:
        return "Error: Cannot divide by zero. Please provide a non-zero divisor."
    return f"{a} / {b} = {a / b:.4f}"


print("good_divide(10, 3):", good_divide.invoke({"a": 10.0, "b": 3.0}))
print("good_divide(10, 0):", good_divide.invoke({"a": 10.0, "b": 0.0}))

print("\nThe bad version would crash on b=0 — try it yourself if you're curious!")

In [ ]:
@tool
def search_products(
    query: str,
    category: Optional[str] = None,
    max_price: Optional[float] = None,
    min_rating: Optional[float] = None,
) -> str:
    """
    Search for products in the catalog.

    Use when asked to find, search, or look for products.
    Can filter by category, maximum price, and minimum rating.
    """
    errors = []
    if not query or len(query.strip()) == 0:
        errors.append("Query cannot be empty")
    if max_price is not None and max_price < 0:
        errors.append("Max price cannot be negative")
    if min_rating is not None and (min_rating < 0 or min_rating > 5):
        errors.append("Rating must be between 0 and 5")
    if errors:
        return f"Validation errors: {'; '.join(errors)}"

    parts = [f"Searching for: '{query.strip()}'"]
    if category:
        parts.append(f"Category: {category}")
    if max_price is not None:
        parts.append(f"Max price: ${max_price:.2f}")
    if min_rating is not None:
        parts.append(f"Min rating: {min_rating}+ stars")
    parts.append("Found 15 products matching your criteria.")
    return " | ".join(parts)


print("Valid search:")
print(search_products.invoke({"query": "laptop", "category": "electronics", "max_price": 1000.0, "min_rating": 4.0}))

print("\nEmpty query:")
print(search_products.invoke({"query": ""}))

print("\nNegative price:")
print(search_products.invoke({"query": "laptop", "max_price": -50.0}))

print("\nInvalid rating:")
print(search_products.invoke({"query": "laptop", "min_rating": 10.0}))

### Tool Testing Checklist

Before plugging a tool into an agent, verify:

- [ ] Normal inputs produce correct results
- [ ] Edge cases are handled (zero, negative, empty, max values)
- [ ] Invalid inputs return helpful error messages — **don't crash**
- [ ] Return values are clear strings the model can relay to the user
- [ ] Docstring accurately describes when to use the tool

In [ ]:
print("--- Integration test: wiring tested tools into an agent ---\n")

shopping_agent = create_agent(
    model="bedrock:us.amazon.nova-2-lite-v1:0",
    tools=[calculate_discount, good_divide, search_products],
    system_prompt=(
        "You are a helpful shopping assistant. "
        "Use the available tools to help users with calculations and product searches. "
        "If a tool returns an error, explain it to the user and ask for correct input."
    ),
    name="shopping_assistant_agent",
)

queries = [
    "What's the price of an $80 item after a 25% discount?",
    "Search for laptops under $500 with at least 4 stars",
]

for q in queries:
    print(f"User:  {q}")
    result = shopping_agent.invoke({"messages": [{"role": "user", "content": q}]})
    print(f"Agent: {result['messages'][-1].content}\n")

---
## Stage 4 — Short-Term Memory & Persistence

Everything we've built so far is **stateless** — each `invoke()` call starts fresh. The agent has no idea what you said 10 seconds ago. In a real conversation, that's a dealbreaker.

LangChain v1.0 solves this with **checkpointers**. A checkpointer saves the conversation state after every turn and reloads it on the next turn.

For development we use `InMemorySaver` from `langgraph.checkpoint.memory`.

> **Deprecated:** `ConversationBufferMemory`, `RunnableWithMessageHistory` — do not use these.

In [ ]:
print("=== Agent WITHOUT Memory ===")
print("Each invoke() is independent — the agent forgets everything.\n")

forgetful_agent = create_agent(
    model="bedrock:us.amazon.nova-2-lite-v1:0",
    tools=[],
    system_prompt="You are a friendly assistant. Remember important details about the user.",
    name="forgetful_agent",
)

r1 = forgetful_agent.invoke(
    {"messages": [{"role": "user", "content": "Hi! My name is Alice and I love programming."}]}
)
print("Turn 1 — User: Hi! My name is Alice and I love programming.")
print("Turn 1 — Agent:", r1["messages"][-1].content)

r2 = forgetful_agent.invoke(
    {"messages": [{"role": "user", "content": "What's my name and what do I love?"}]}
)
print("\nTurn 2 — User: What's my name and what do I love?")
print("Turn 2 — Agent:", r2["messages"][-1].content)
print("\n^^^ The agent forgot! It has no memory of the previous turn.")

### Adding Memory with `InMemorySaver`

Two things change:

1. Pass `checkpointer=InMemorySaver()` when creating the agent.
2. Pass a **config** with a `thread_id` on every `invoke()` call.

```python
config = {"configurable": {"thread_id": "some_unique_id"}}
```

Same `thread_id` = same conversation = shared memory.  
Different `thread_id` = separate conversation = isolated memory.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory_agent = create_agent(
    model="bedrock:us.amazon.nova-2-lite-v1:0",
    tools=[],
    system_prompt="You are a friendly assistant. Remember important details about the user.",
    checkpointer=InMemorySaver(),
    name="memory_agent",
)

config = {"configurable": {"thread_id": "user_alice_123"}}

r1 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "Hi! My name is Alice and I love programming."}]},
    config,
)
print("Turn 1 — User: Hi! My name is Alice and I love programming.")
print("Turn 1 — Agent:", r1["messages"][-1].content)

r2 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "What's my name and what do I love?"}]},
    config,
)
print("\nTurn 2 — User: What's my name and what do I love?")
print("Turn 2 — Agent:", r2["messages"][-1].content)

r3 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "Summarise everything you know about me."}]},
    config,
)
print("\nTurn 3 — User: Summarise everything you know about me.")
print("Turn 3 — Agent:", r3["messages"][-1].content)

### Thread-Based Conversations

Different `thread_id` values create **completely isolated** conversations. What happens in Thread A stays in Thread A. This is how you support multiple users with a single agent instance.

Common `thread_id` patterns:

| Pattern | Example | Use case |
|---------|---------|----------|
| User-based | `f"user_{user_id}"` | One persistent conversation per user |
| Session-based | `f"user_{user_id}_{uuid}"` | New thread each session |
| Topic-based | `f"user_{user_id}_{topic}"` | Separate threads per topic |

In [ ]:
support_agent = create_agent(
    model="bedrock:us.amazon.nova-2-lite-v1:0",
    tools=[],
    system_prompt="You are a customer support agent. Remember the customer's name and issue.",
    checkpointer=InMemorySaver(),
    name="support_agent",
)

thread_alice = {"configurable": {"thread_id": "user_alice_session"}}
thread_bob = {"configurable": {"thread_id": "user_bob_session"}}

print("--- Alice starts ---")
r = support_agent.invoke(
    {"messages": [{"role": "user", "content": "Hi, I'm Alice. I have a billing question."}]},
    thread_alice,
)
print("Alice:", r["messages"][-1].content)

print("\n--- Bob starts ---")
r = support_agent.invoke(
    {"messages": [{"role": "user", "content": "Hello, my name is Bob. I need help with a return."}]},
    thread_bob,
)
print("Bob:", r["messages"][-1].content)

print("\n--- Alice follows up (her context is preserved) ---")
r = support_agent.invoke(
    {"messages": [{"role": "user", "content": "What was my issue again?"}]},
    thread_alice,
)
print("Alice:", r["messages"][-1].content)

print("\n--- Bob follows up (his context is separate) ---")
r = support_agent.invoke(
    {"messages": [{"role": "user", "content": "What's my name and what did I need help with?"}]},
    thread_bob,
)
print("Bob:", r["messages"][-1].content)

### Inspecting Agent State

The result from `invoke()` contains the full message history under `result["messages"]`. Each message has a type (`human`, `ai`, `tool`) and content. Inspecting state is invaluable for debugging.

**Common issues to look for:**

| Symptom | Check |
|---------|-------|
| Agent doesn't remember | Is `thread_id` consistent? Is checkpointer configured? |
| Wrong tool called | Inspect `tool_calls` on the `AIMessage` |
| Conversation too long | Count total messages — more messages = more tokens = higher cost |

In [ ]:
@tool
def get_product_info(product_id: str) -> str:
    """Look up product information by ID."""
    products = {
        "P001": "Widget Pro - $49.99 - In Stock",
        "P002": "Gadget Plus - $79.99 - Low Stock",
        "P003": "Super Tool - $129.99 - Out of Stock",
    }
    return products.get(product_id, f"Product {product_id} not found")


debug_agent = create_agent(
    model="bedrock:us.amazon.nova-2-lite-v1:0",
    tools=[get_product_info],
    system_prompt="You help customers find product information.",
    checkpointer=InMemorySaver(),
    name="debug_demo_agent",
)

cfg = {"configurable": {"thread_id": "debug_session_001"}}

debug_agent.invoke(
    {"messages": [{"role": "user", "content": "What's the price of product P001?"}]}, cfg
)
result = debug_agent.invoke(
    {"messages": [{"role": "user", "content": "What about P002?"}]}, cfg
)

print(f"Total messages in state: {len(result['messages'])}\n")

for i, msg in enumerate(result["messages"]):
    msg_type = getattr(msg, "type", type(msg).__name__)
    content_preview = str(msg.content)[:80] if hasattr(msg, "content") else ""
    tool_info = ""
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        tool_info = f"  [calls: {[tc['name'] for tc in msg.tool_calls]}]"
    print(f"  {i+1:2}. {msg_type:8}{tool_info}")
    if content_preview:
        print(f"      {content_preview}")

### Memory with Tools — Full Example

Memory and tools work together seamlessly. The agent remembers earlier tool results and can reference them in later turns.

In [ ]:
@tool
def get_travel_weather(city: str) -> str:
    """Get weather for a city. Use when asked about weather."""
    weather = {
        "austin": "Sunny, 85°F",
        "seattle": "Rainy, 55°F",
        "new york": "Cloudy, 68°F",
    }
    return weather.get(city.lower(), f"Weather for {city}: Partly cloudy, 70°F")


travel_agent = create_agent(
    model="bedrock:us.amazon.nova-2-lite-v1:0",
    tools=[get_travel_weather],
    system_prompt=(
        "You are a travel assistant. "
        "Remember user preferences and destinations they mention. "
        "Use the weather tool when asked about weather."
    ),
    checkpointer=InMemorySaver(),
    name="travel_assistant",
)

trip_config = {"configurable": {"thread_id": "user_bob_trip"}}

turns = [
    "I'm planning a trip from Austin to Seattle next week.",
    "What's the weather like in my destination?",
    "And what about where I'm leaving from?",
    "What was my original travel plan again?",
]

for i, message in enumerate(turns, 1):
    result = travel_agent.invoke(
        {"messages": [{"role": "user", "content": message}]}, trip_config
    )
    response = result["messages"][-1].content
    print(f"Turn {i} — User: {message}")
    print(f"         Agent: {response[:250]}")
    print()

### State vs. Store — A Preview

Everything we've used today is **state** (short-term, session-scoped memory). LangChain v1.0 also supports **store** (permanent, cross-conversation memory) for things like user preferences that should persist forever.

| Aspect | State (today) | Store (later) |
|--------|---------------|---------------|
| Scope | Single conversation thread | Cross-conversation |
| Lifetime | Until thread ends | Permanent |
| Setup | Automatic via checkpointer | Manual read/write in tools |
| Use case | Message history | User profiles, learned facts |

For development, `InMemorySaver` is perfect. For production, swap in `PostgresSaver` or `SqliteSaver` — the code stays exactly the same.

---
## Key Takeaways

1. **`@tool` turns functions into capabilities** — the decorator + a good docstring is all you need.
2. **Docstrings are routing instructions** — the agent never sees your code, only the name and description.
3. **`create_agent()` is the v1.0 standard** — pass `model`, `tools`, `name`, and optionally `system_prompt`.
4. **Always test tools before integration** — use `.invoke()` with normal, edge-case, and error inputs.
5. **`InMemorySaver` adds conversation memory** — just add `checkpointer=InMemorySaver()` and a `thread_id`.
6. **`thread_id` isolates conversations** — different IDs = separate memory spaces.
7. **Inspect state for debugging** — `result["messages"]` shows the full conversation including tool calls.

## Up Next — Friday

Tomorrow we'll explore **Structured Output & RAG**: how to force the model to return data in a specific schema (Pydantic models) and how to connect agents to external knowledge via retrieval-augmented generation.